# ARI 410/510 Lab 2 Starter Notebook

This notebook only gets the data loaded for you: it downloads and unzips the dataset, reads the provided train and test files, and attaches the activity names and feature names. Everything after that is the lab.

**How to use this:** In Colab, *File → Save a copy in Drive*, then run top to bottom.

## 0. Get the data

Either download it directly (next cell), or download the zip from Canvas and upload it to Colab's file panel (left sidebar), then set `ZIP_PATH` to its name.

Note: the download is a zip *inside* a zip. The helper below handles that for you.

In [7]:
import os, zipfile, glob, urllib.request

ZIP_PATH = "UCI HAR Dataset.zip"   # change this if you uploaded the file under a different name
URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

if not os.path.exists(ZIP_PATH):
    try:
        print("Downloading from UCI...")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    except Exception as e:
        print("Download failed:", e, "\nUpload the zip from Canvas instead and set ZIP_PATH.")

# Unzip the outer zip, then the inner 'UCI HAR Dataset.zip'
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("har_raw")
inner = glob.glob("har_raw/**/UCI HAR Dataset.zip", recursive=True)
if inner:
    with zipfile.ZipFile(inner[0]) as z:
        z.extractall("har_raw")
DATA = glob.glob("har_raw/**/UCI HAR Dataset/train", recursive=True)[0].rsplit("train", 1)[0] + "\\"
print("Data folder:", DATA)

Data folder: har_raw\UCI HAR Dataset\\


## 1. Load the data

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Feature names. Some names appear more than once in features.txt, so we add
# a numeric suffix to duplicates to keep every name unique.
raw_names = pd.read_csv(DATA + "features.txt", sep=r"\s+", header=None)[1].tolist()
seen, feature_names = {}, []
for n in raw_names:
    seen[n] = seen.get(n, 0) + 1
    feature_names.append(n if seen[n] == 1 else f"{n}__{seen[n]}")

# Map integer labels (1-6) to activity names
label_map = dict(pd.read_csv(DATA + "activity_labels.txt", sep=r"\s+", header=None).values)

def load(split):
    X = pd.read_csv(DATA + f"{split}/X_{split}.txt", sep=r"\s+", header=None).values
    y = pd.read_csv(DATA + f"{split}/y_{split}.txt", header=None)[0].map(label_map).values
    return X, y

X_train_full, y_train_full = load("train")
X_test, y_test = load("test")
print("train:", X_train_full.shape, " test:", X_test.shape)
print("activities:", sorted(set(y_train_full)))

train: (7352, 561)  test: (2947, 561)
activities: ['LAYING', 'SITTING', 'STANDING', 'WALKING', 'WALKING_DOWNSTAIRS', 'WALKING_UPSTAIRS']


## Plotting helpers

Making plots isn't the point of this lab, so here are three general-purpose helpers. They're demonstrated below on **made-up toy data**; you'll call them with your own results. Feel free to modify them or write your own.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_bar(labels, title="", xlabel="", ylabel="count"):
    """Bar chart of how often each value in `labels` occurs."""
    counts = pd.Series(labels).value_counts()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    counts.plot(kind="bar", ax=ax)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

def plot_lines(x, series, title="", xlabel="", ylabel="", hline=None, hline_label=None, logx=False):
    """Line plot of one or more series against x.
    `series` is a dict {line name: list of y values (same length as x)}.
    Optionally draws a dashed horizontal reference line at y = hline."""
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, ys in series.items():
        ax.plot(x, ys, marker="o", label=name)
    if hline is not None:
        ax.axhline(hline, linestyle="--", color="gray", label=hline_label)
    if logx:
        ax.set_xscale("log")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if len(series) > 1 or hline_label:
        ax.legend()
    plt.tight_layout(); plt.show()

def scatter_2d(Z, labels, title="", xlabel="", ylabel=""):
    """Scatter plot of the first two columns of Z, one color per value in `labels`."""
    Z, labels = np.asarray(Z), np.asarray(labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    for lab in sorted(set(labels), key=str):
        m = labels == lab
        ax.scatter(Z[m, 0], Z[m, 1], s=5, alpha=0.5, label=str(lab))
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.legend(markerscale=3, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.show()

In [ ]:
# ---- Demo on made-up toy data. Replace with your own data and results. ----
rng = np.random.default_rng(0)

# plot_bar: pass a list/array of category labels
# create random data
toy_labels = rng.choice(["apple", "banana", "cherry"], size=200, p=[0.5, 0.3, 0.2])
# plot it
plot_bar(toy_labels, title="Toy example: fruit counts")

# plot_lines: x values plus a dict of named y-value lists
toy_x = [1, 2, 3, 4, 5]
plot_lines(toy_x, {"method A": [0.2, 0.5, 0.7, 0.8, 0.85],
                   "method B": [0.4, 0.6, 0.65, 0.7, 0.72]},
           title="Toy example: two lines", xlabel="x", ylabel="score",
           hline=0.9, hline_label="reference")

# scatter_2d: an (n, 2+) array plus one label per row
toy_points = np.vstack([rng.normal([0, 0], 1, (100, 2)), rng.normal([4, 3], 1, (100, 2))])
toy_groups = ["group 1"] * 100 + ["group 2"] * 100
scatter_2d(toy_points, toy_groups, title="Toy example: two groups", xlabel="dim 1", ylabel="dim 2")

## You take it from here

You now have `X_train_full`, `y_train_full` (the provided training set), `X_test`, `y_test`, (the held-out test set) and `feature_names`. The assignment PDF describes what the lab needs to cover. How you organize your code and your notebook is up to you; add as many cells as you like, or don't use this starter notebook at all if you don't want to.

In [13]:
# split dev set

from sklearn.model_selection import train_test_split

X_train, X_dev, y_train, y_dev = train_test_split(X_train_full, y_train_full, test_size = .8, random_state=42)

In [14]:
# scale test and dev set

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)